# TranscriptIQ — exploratory analysis

**Business question.** What do customers and internal teams talk about, how does sentiment differ by call type, and which calls point to renewal risk, feature demand or engineering blockers?

**How to read this notebook.** It reads the output of `python main.py run ...` (nothing is hardcoded). Set `RUN` below to `demo` (heuristic, no API key) or `llm`.
If the run used the synthetic dataset, **every number here describes invented data**, not a company.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import plotly.express as px
try:
    from IPython.display import display
except ImportError:
    display = print
from src import analytics, storage
from src.evaluation import compute_ground_truth_metrics, compute_metrics, load_ground_truth, load_review

RUN = "demo"   # "demo" or "llm"
data = storage.load_processed(ROOT / "data" / "processed" / RUN)
assert data is not None, f"No processed data for run '{RUN}'. Run: python main.py run --mode {RUN}"
df, meta, quality = data["df"], data["meta"], data["quality"]
print(meta["disclaimer"])
print({k: meta[k] for k in ("records_loaded", "records_analyzable", "failed", "runtime_seconds")})

## 1. Data inspection
We look at what we actually have before trusting any analysis: record counts per call type, data origin and a few raw transcripts. Origins are never mixed: original and synthetic records must be analyzed separately.

In [ ]:
print(df["data_origin"].value_counts().to_dict())
print(df["call_type"].value_counts().to_dict())
display(df[["transcript_id", "call_type", "call_date", "n_words", "n_turns"]].head())
print(df.iloc[0]["transcript_text"])

## 2. Data quality
Why first: duplicates inflate topic counts, empty transcripts bias sentiment toward neutral, and missing dates make trend charts misleading. Excluded records are reported, not silently dropped.

In [ ]:
rows = [{"check": k, "count": v["count"], "examples": v["examples"]} for k, v in quality.items() if isinstance(v, dict) and "count" in v]
display(pd.DataFrame(rows))

## 3. Exploratory clusters
Why: the taxonomy should be grounded in the data, not just in assumptions. Clusters (embeddings + KMeans) are a *discovery* tool. Silhouette picks a default k, but with ~100 documents it is a weak guide, so we read the representative transcripts for each cluster before deciding what they mean.

In [ ]:
cl = data["clusters"] or {}
print(cl.get("method"), "k =", cl.get("k"), cl.get("silhouette_by_k"))
for c in cl.get("clusters", []):
    print(f"\nCluster {c['cluster']} (n={c['size']}) terms: {', '.join(c['top_terms'][:6])} | mix: {c['call_type_mix']}")
    for r in c["representatives"][:2]:
        print("  ", r["transcript_id"], r["excerpt"][:140].replace("\n", " "))

## 4. Taxonomy decisions
We start from the candidate taxonomy in `config/taxonomy.json`, then compare it with the clusters: topics with no matching cluster are candidates to merge or drop; clusters that map to no topic are candidates to add.

**Trade-offs.** *Rules*: transparent and free, but brittle with synonyms and overlapping topics. *Clustering*: finds unanticipated themes, but clusters are unlabeled and unstable on small samples. *LLM*: handles nuance and overlap, but costs money, can drift, and so is constrained to the allowed list and validated (unknown names become "Other").

In [ ]:
display(analytics.topic_counts(df))
fig = px.imshow(analytics.topic_by_call_type(df), text_auto=True, aspect="auto", title="Topics by call type (counts)")
fig.show()
display(analytics.topic_by_call_type_pct(df))

## 5. Sentiment by call type and topic
Percentages use the size of the group as denominator (e.g. negative support calls / all analyzed support calls). Customer sentiment is separate from overall tone and is empty for internal calls. Scores are **not calibrated probabilities**.

In [ ]:
sct = analytics.sentiment_by_group(df, "call_type"); display(sct)
print(analytics.describe_group_sentiment(sct, "call_type"))
stp = analytics.sentiment_by_group(df, "primary_topic"); display(stp)
print(analytics.describe_group_sentiment(stp, "primary_topic"))
display(analytics.negative_concentration(df, "primary_topic"))
trend = analytics.sentiment_trend(df)
print("Trend not shown: dates unreliable." if trend is None else "")
if trend is not None:
    px.line(trend, x="month", y="negative_pct", markers=True).show()

**Interpretation guide.** Read the call-type table first, then the topic table. Differences between call types partly reflect *what is discussed* (a support call is about a problem, a renewal call about money), so compare topics within a call type before concluding anything. Small groups (n<10) are flagged.

In [ ]:
cust = analytics.sentiment_by_group(df, "call_type", "customer_sentiment", "customer_sentiment_score")
display(cust)

## 6. Stakeholder insights
Each insight has a metric (computed in pandas), transcript IDs, evidence quotes, an interpretation, a recommended action and a caveat. Renewal items are **signals**, not churn predictions; cross-team links are **candidate matches** from text similarity.

In [ ]:
for ins in analytics.build_insights(df):
    print(f"\n[{ins['stakeholder']}] {ins['title']}\n  metric: {ins['metric']}\n  action: {ins['recommended_action']}\n  caveat: {ins['caveat']}")
    for e in ins["evidence"][:3]:
        print("   -", e["transcript_id"], "|", e["quote"][:120])
display(analytics.feature_group_table(df)[["representative", "n_transcripts"]].head(10))

## 7. Evaluation
Synthetic runs can be compared with the separate scenario-designed labels in `data/synthetic/ground_truth.csv`; these are not human-reviewed and do not estimate real-world accuracy. For an independent check, run `python main.py review-sample`, fill in the reviewer columns of `review_sheet.csv` (20–30 transcripts), then re-run this cell.

In [ ]:
synthetic = df[df["data_origin"] == "synthetic"]
ground_truth = load_ground_truth(ROOT / "data" / "synthetic" / "ground_truth.csv")
if not synthetic.empty and ground_truth is not None:
    truth_metrics = compute_ground_truth_metrics(synthetic, ground_truth)
    print(json.dumps(truth_metrics, indent=2) if truth_metrics else "No synthetic labels matched this run.")
review = load_review(ROOT / "data" / "processed" / RUN / "review_sheet.csv")
metrics = compute_metrics(review) if review is not None else None
print(json.dumps(metrics, indent=2) if metrics else "No human-reviewed labels yet: no human-review metrics computed.")

## 8. Caveats and recommendations
**Caveats.** Small sample; uneven call types; uncalibrated sentiment; sarcasm and mixed tone are error-prone; renewal items are signals, not predictions; cross-team links are candidates; PII redaction does not cover names; demo mode is rules + lexicon, not an LLM; and synthetic data is invented.

**Recommendations.** Use the stakeholder insights above as a starting list for Customer Success, Product and Engineering, but confirm the top items by reading the linked transcripts first. Before acting at scale, run the LLM mode on real transcripts and measure precision on a reviewed sample.